# 04 - ML scores + Customer 360 (Gold)

Computes churn risk and cross-sell recommendations from the curated tables,
then builds the denormalized `customer_360` gold table used by the SQL endpoint
and the Fabric Data Agent. All tables are read from / written to the **`gold`** schema.

## Use the gold schema

Sets the current schema to `gold` so the statements below read the curated tables
and create the ML + `customer_360` tables in `gold`.

In [ ]:
spark.sql('CREATE SCHEMA IF NOT EXISTS gold')
spark.sql('USE gold')
print('current schema:', spark.catalog.currentDatabase())

## Churn score (rule-based, transparent heuristic)

In [ ]:
spark.sql('''CREATE OR REPLACE TABLE ml_churn_score AS
WITH unpaid AS (
    SELECT account_id, COUNT(*) AS unpaid_ct
    FROM fact_invoice WHERE paid = false GROUP BY account_id
),
uptime AS (
    SELECT account_id, AVG(uptime_pct) AS avg_uptime
    FROM fact_service_metric GROUP BY account_id
),
csat AS (
    SELECT account_id, AVG(csat) AS avg_csat
    FROM fact_feedback GROUP BY account_id
),
cancels AS (
    SELECT customer_id, COUNT(*) AS cancel_ct
    FROM fact_contact WHERE reason = 'cancel_request' GROUP BY customer_id
),
scored AS (
    SELECT
        a.customer_id, a.account_id,
        LEAST(0.98, GREATEST(0.01,
            0.10
            + CASE WHEN a.status = 'suspended' THEN 0.25 ELSE 0 END
            + LEAST(0.30, 0.10 * COALESCE(u.unpaid_ct, 0))
            + CASE WHEN COALESCE(up.avg_uptime, 100) < 99.0 THEN 0.20 ELSE 0 END
            + CASE WHEN COALESCE(cs.avg_csat, 5) <= 2.5 THEN 0.20 ELSE 0 END
            + CASE WHEN COALESCE(cn.cancel_ct, 0) > 0 THEN 0.25 ELSE 0 END
            + CASE WHEN c.tenure_months < 6 THEN 0.10
                   WHEN c.tenure_months > 48 THEN -0.05 ELSE 0 END
            + CASE WHEN a.autopay = false THEN 0.05 ELSE 0 END
        )) AS churn_probability,
        CASE
            WHEN COALESCE(cn.cancel_ct,0) > 0 THEN 'cancellation intent'
            WHEN a.status = 'suspended' THEN 'account suspended'
            WHEN COALESCE(up.avg_uptime,100) < 99.0 THEN 'degraded service'
            WHEN COALESCE(cs.avg_csat,5) <= 2.5 THEN 'low satisfaction'
            WHEN COALESCE(u.unpaid_ct,0) > 0 THEN 'unpaid invoices'
            WHEN c.tenure_months < 6 THEN 'new/short tenure'
            ELSE 'stable account'
        END AS top_reason
    FROM dim_account a
    JOIN dim_customer c ON a.customer_id = c.customer_id
    LEFT JOIN unpaid u ON a.account_id = u.account_id
    LEFT JOIN uptime up ON a.account_id = up.account_id
    LEFT JOIN csat cs ON a.account_id = cs.account_id
    LEFT JOIN cancels cn ON a.customer_id = cn.customer_id
)
SELECT customer_id, account_id,
       ROUND(churn_probability, 3) AS churn_probability,
       CASE WHEN churn_probability >= 0.55 THEN 'High'
            WHEN churn_probability >= 0.30 THEN 'Medium' ELSE 'Low' END AS risk_band,
       top_reason,
       CURRENT_DATE() AS scored_date
FROM scored''')
print('ml_churn_score rows:', spark.table('ml_churn_score').count())
spark.sql('SELECT risk_band, COUNT(*) c FROM ml_churn_score GROUP BY risk_band').show()

## Cross-sell recommendations (product-gap logic)

In [ ]:
spark.sql('''CREATE OR REPLACE TABLE ml_crosssell_reco AS
WITH owned AS (
    SELECT account_id, collect_set(product_id) AS products
    FROM fact_subscription GROUP BY account_id
)
SELECT
    a.account_id,
    CASE WHEN NOT array_contains(o.products, 'PROD_MOB') THEN 'PROD_MOB'
         WHEN NOT array_contains(o.products, 'PROD_TV')  THEN 'PROD_TV'
         ELSE 'PROD_VOICE' END AS recommended_product_id,
    CASE WHEN NOT array_contains(o.products, 'PROD_MOB') THEN 'PROMO_XSELL_1'
         WHEN NOT array_contains(o.products, 'PROD_TV')  THEN 'PROMO_XSELL_2'
         ELSE NULL END AS recommended_promotion_id,
    ROUND(0.4 + LEAST(c.tenure_months, 60) / 200.0, 3) AS score,
    CASE WHEN NOT array_contains(o.products, 'PROD_MOB')
              THEN 'Internet customer without mobile'
         WHEN NOT array_contains(o.products, 'PROD_TV')
              THEN 'Eligible for TV + internet bundle'
         ELSE 'Add home phone to bundle' END AS rationale,
    CURRENT_DATE() AS scored_date
FROM dim_account a
JOIN dim_customer c ON a.customer_id = c.customer_id
JOIN owned o ON a.account_id = o.account_id
WHERE a.status <> 'cancelled'
  AND NOT (array_contains(o.products, 'PROD_MOB')
           AND array_contains(o.products, 'PROD_TV')
           AND array_contains(o.products, 'PROD_VOICE'))''')
print('ml_crosssell_reco rows:', spark.table('ml_crosssell_reco').count())
spark.sql('SELECT recommended_product_id, COUNT(*) c FROM ml_crosssell_reco GROUP BY recommended_product_id').show()

## Customer 360 (gold serving object)

In [ ]:
spark.sql('''CREATE OR REPLACE TABLE customer_360 AS
WITH sub_agg AS (
    SELECT account_id,
           COUNT(*) AS active_products,
           ROUND(SUM(mrc), 2) AS total_mrc,
           concat_ws(', ', collect_list(product_id)) AS product_list
    FROM fact_subscription WHERE status = 'active' GROUP BY account_id
),
latest_inv AS (
    SELECT account_id, period_end, amount, due_date, paid, is_first_bill
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY period_end DESC) rn
        FROM fact_invoice
    ) WHERE rn = 1
),
balance AS (
    SELECT account_id, ROUND(SUM(amount), 2) AS open_balance
    FROM fact_invoice WHERE paid = false GROUP BY account_id
),
open_wo AS (
    SELECT account_id, COUNT(*) AS open_work_orders
    FROM fact_work_order WHERE status = 'open' GROUP BY account_id
),
last_contact AS (
    SELECT customer_id, MAX(contact_ts) AS last_contact_ts
    FROM fact_contact GROUP BY customer_id
),
recent_outage AS (
    SELECT DISTINCT geo_id FROM fact_outage
    WHERE start_time >= date_sub(current_timestamp(), 14)
)
SELECT
    c.customer_id, a.account_id,
    c.first_name, c.last_name, c.email, c.phone, c.segment,
    c.tenure_months, a.status AS account_status, a.open_date, a.is_new_customer,
    a.autopay, a.credit_class,
    g.city, g.state, g.region, g.zip,
    COALESCE(s.active_products, 0) AS active_products,
    COALESCE(s.total_mrc, 0) AS total_mrc,
    s.product_list,
    li.amount AS last_invoice_amount, li.due_date AS last_invoice_due,
    li.paid AS last_invoice_paid, li.is_first_bill AS last_invoice_is_first_bill,
    COALESCE(b.open_balance, 0) AS open_balance,
    COALESCE(w.open_work_orders, 0) AS open_work_orders,
    lc.last_contact_ts,
    CASE WHEN ro.geo_id IS NOT NULL THEN true ELSE false END AS recent_outage_exposure,
    ch.churn_probability, ch.risk_band, ch.top_reason AS churn_top_reason,
    xs.recommended_product_id AS top_crosssell_product,
    xs.recommended_promotion_id AS top_crosssell_promo,
    xs.score AS top_crosssell_score
FROM dim_customer c
JOIN dim_account a ON c.customer_id = a.customer_id
LEFT JOIN dim_geography g ON c.geo_id = g.geo_id
LEFT JOIN sub_agg s ON a.account_id = s.account_id
LEFT JOIN latest_inv li ON a.account_id = li.account_id
LEFT JOIN balance b ON a.account_id = b.account_id
LEFT JOIN open_wo w ON a.account_id = w.account_id
LEFT JOIN last_contact lc ON c.customer_id = lc.customer_id
LEFT JOIN recent_outage ro ON c.geo_id = ro.geo_id
LEFT JOIN ml_churn_score ch ON c.customer_id = ch.customer_id
LEFT JOIN ml_crosssell_reco xs ON a.account_id = xs.account_id''')
print('customer_360 rows:', spark.table('customer_360').count())

## Validation: sample a new customer with an unpaid first bill

In [ ]:
spark.sql('''
SELECT customer_id, first_name, last_name, account_status, tenure_months,
       last_invoice_amount, last_invoice_is_first_bill, open_balance,
       risk_band, top_crosssell_product
FROM customer_360
WHERE last_invoice_is_first_bill = true AND last_invoice_paid = false
LIMIT 10''').show(truncate=False)